# Respiratory CDSS EDA

Bu notebook diplom uchun bazaviy EDA oqimini beradi. Hozirgi versiya seed dataset, data quality report, dataset profile va model evaluation artefaktlarini bir joyda ko'rish uchun ishlatiladi.

Maqsadlar:
- dataset hajmi va ustunlarini ko'rish
- label distribution va missing holatini tekshirish
- numeric feature summary olish
- model holdout va cross-validation natijalarini bir joyda ko'rish


In [ ]:
from __future__ import annotations

import csv
import json
from collections import Counter
from pathlib import Path
from pprint import pprint
from statistics import mean


def find_backend_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "app").exists() and (candidate / "data").exists():
            return candidate
        backend_candidate = candidate / "backend"
        if (backend_candidate / "app").exists() and (backend_candidate / "data").exists():
            return backend_candidate
    raise RuntimeError("Backend root topilmadi")


BACKEND_ROOT = find_backend_root(Path.cwd())
DATA_DIR = BACKEND_ROOT / "data"
MODEL_DIR = BACKEND_ROOT / "ml_models"

RAW_DATASET_PATH = DATA_DIR / "respiratory_seed_cases.csv"
CANONICAL_DATASET_PATH = DATA_DIR / "respiratory_canonical_dataset.csv"
FEATURE_DATASET_PATH = DATA_DIR / "respiratory_feature_dataset.csv"
PROFILE_PATH = DATA_DIR / "respiratory_seed_profile.json"
QUALITY_PATH = DATA_DIR / "respiratory_data_quality.json"
CLEANING_PATH = DATA_DIR / "respiratory_cleaning_report.json"
VALIDATION_PATH = DATA_DIR / "real_dataset_validation.json"
METRICS_PATH = MODEL_DIR / "respiratory_nb_metrics.json"
EVALUATION_PATH = MODEL_DIR / "respiratory_nb_evaluation.json"

print(f"Backend root: {BACKEND_ROOT}")


In [ ]:
def load_csv_rows(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8", newline="") as csv_file:
        return list(csv.DictReader(csv_file))


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as json_file:
        return json.load(json_file)


raw_rows = load_csv_rows(RAW_DATASET_PATH)
canonical_rows = load_csv_rows(CANONICAL_DATASET_PATH)
feature_rows = load_csv_rows(FEATURE_DATASET_PATH)

profile = load_json(PROFILE_PATH)
quality = load_json(QUALITY_PATH)
cleaning = load_json(CLEANING_PATH)
validation = load_json(VALIDATION_PATH)
metrics = load_json(METRICS_PATH)
evaluation = load_json(EVALUATION_PATH)

print("Artifacts loaded successfully")
print(f"Raw rows: {len(raw_rows)}")
print(f"Canonical rows: {len(canonical_rows)}")
print(f"Feature rows: {len(feature_rows)}")


In [ ]:
print("Raw dataset columns:")
print(list(raw_rows[0].keys()))

print("\nFirst 5 raw rows:")
for row in raw_rows[:5]:
    pprint(row)


In [ ]:
label_distribution = Counter(row["diagnosis_label"] for row in canonical_rows)
max_count = max(label_distribution.values()) if label_distribution else 1


def ascii_bar(count: int, width: int = 24) -> str:
    scaled = max(1, round((count / max_count) * width)) if count else 0
    return "#" * scaled


print("Label distribution:")
for label, count in sorted(label_distribution.items()):
    print(f"{label:28} | {count:2} | {ascii_bar(count)}")

print("\nMissing counts from quality report:")
for field, count in quality["missing_counts"].items():
    if count > 0:
        print(f"- {field}: {count}")

print("\nQuality warnings:")
for item in quality["warnings"]:
    print(f"- {item}")


In [ ]:
print("Cleaning summary:")
print(f"- Rows with any change: {cleaning['rows_with_any_change']}")
print(f"- Row change rate: {cleaning['row_change_rate']}")
print(f"- Total field changes: {cleaning['total_field_changes']}")

print("\nChanged fields:")
if cleaning['changed_field_counts']:
    for field, count in cleaning['changed_field_counts'].items():
        print(f"- {field}: {count}")
else:
    print("- No field-level changes detected")

print("\nDefault-filled fields:")
if cleaning['defaulted_field_counts']:
    for field, count in cleaning['defaulted_field_counts'].items():
        print(f"- {field}: {count}")
else:
    print("- No defaults applied")


In [ ]:
numeric_fields = [
    "temperature",
    "fatigue_level",
    "duration_days",
    "oxygen_saturation",
    "heart_rate",
    "respiratory_rate",
]

print("Numeric summary from canonical dataset:")
for field in numeric_fields:
    values = []
    for row in canonical_rows:
        value = row.get(field, "")
        if value == "":
            continue
        values.append(float(value))
    print(
        f"- {field}: count={len(values)}, min={min(values):.2f}, max={max(values):.2f}, mean={mean(values):.2f}"
    )


In [ ]:
print("Profile summary:")
print(f"- Total rows: {profile['total_rows']}")
print(f"- Total labels: {profile['total_labels']}")
print(f"- Feature count: {len(profile['feature_distribution'])}")

print("\nTop feature distributions:")
for feature_name in ["temperature_bin", "cough_type", "dyspnea_level", "oxygen_bin"]:
    print(f"\n{feature_name}")
    pprint(profile["feature_distribution"][feature_name])


In [ ]:
print("Holdout metrics:")
print(f"- Train samples: {metrics['train_samples']}")
print(f"- Test samples: {metrics['test_samples']}")
print(f"- Split source: {metrics['split_source']}")
print(f"- Accuracy: {metrics['metrics']['accuracy']}")

print("\nCross-validation metrics:")
print(f"- Folds: {evaluation['folds']}")
print(f"- Samples: {evaluation['samples']}")
print(f"- Mean accuracy: {evaluation['mean_accuracy']}")
print(f"- Overall accuracy: {evaluation['overall_accuracy']}")

print("\nFold results:")
for item in evaluation['fold_results']:
    print(
        f"- Fold {item['fold']}: train={item['train_samples']}, test={item['test_samples']}, accuracy={item['accuracy']}"
    )

print("\nPer-label CV accuracy:")
for label, score in evaluation['per_label_accuracy'].items():
    print(f"- {label}: {score}")


In [ ]:
print("Onboarding validation summary:")
print(f"- Ready for pipeline: {validation['ready_for_pipeline']}")
print(f"- Explicit mappings: {validation['explicit_mappings']}")
print(f"- Alias mappings: {validation['alias_mappings']}")
print(f"- Missing required fields: {len(validation['missing_required_fields'])}")
print(f"- Unused dataset columns: {len(validation['unused_dataset_columns'])}")

if validation['missing_required_fields']:
    print("\nMissing required fields:")
    for field in validation['missing_required_fields']:
        print(f"- {field}")


## Real Dataset Onboarding

Seed dataset o'rniga real dataset kelganda quyidagi oqim tavsiya etiladi:

1. `../data/real_dataset_mapping_template.json` ni to'ldiring.
2. `python scripts/validate_real_dataset.py /absolute/path/to/real_dataset.csv /absolute/path/to/mapping.json` ni ishga tushiring.
3. `python scripts/generate_cleaning_report.py /absolute/path/to/real_dataset.csv /absolute/path/to/mapping.json` bilan preprocessing o'zgarishlarini ko'ring.
4. `python scripts/run_ml_pipeline.py /absolute/path/to/real_dataset.csv /absolute/path/to/mapping.json` ni ishga tushiring.
5. Shu notebookni qayta ochib yangi artefaktlar asosida tahlil qiling.


In [ ]:
mapping_template_path = DATA_DIR / "real_dataset_mapping_template.json"
mapping_template = load_json(mapping_template_path)

print(f"Mapping template path: {mapping_template_path}")
print(f"Fields to map: {len(mapping_template['fields'])}")
print("\nFirst 5 canonical fields:")
for item in mapping_template['fields'][:5]:
    print(
        f"- {item['canonical_field']} | required={item['required']} | aliases={', '.join(item['accepted_aliases'])}"
    )

real_dataset_example = "/absolute/path/to/real_dataset.csv"
print("\nCLI examples:")
mapping_example = "/absolute/path/to/mapping.json"
print(f"python scripts/validate_real_dataset.py {real_dataset_example} {mapping_example}")
print(f"python scripts/generate_cleaning_report.py {real_dataset_example} {mapping_example}")
print(f"python scripts/run_ml_pipeline.py {real_dataset_example} {mapping_example}")
print(f"python scripts/generate_dataset_profile.py {real_dataset_example} {mapping_example}")
print(f"python scripts/export_feature_dataset.py {real_dataset_example} {mapping_example}")


## Diplom uchun keyingi kengaytirishlar

1. Real dataset kelganda shu notebookga ustun mapping jadvali qo'shing.
2. Outlier va missing cleaning bosqichlarini oldin/ keyin taqqoslang.
3. Class imbalance kuchaysa resampling yoki class-weight tajribalarini alohida bo'limga ajrating.
4. Baseline NB modelni keyin XGBoost yoki Logistic Regression bilan taqqoslang.
5. Yakuniy diplom matni uchun shu notebookdagi jadval va xulosalarni rasm/diagramma ko'rinishiga olib chiqing.
